In [1]:
import pandas as pd
import numpy as np
import time
import gc
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

print("🚀 CHIẾN DỊCH CHUỘC LỖI: FULL FEATURES + CHỐNG OOM TỐI ƯU 🚀")
start_time = time.time()

PATH_TRAIN = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data/train_clean.parquet'
PATH_TEST = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data/test_clean.parquet'

# ==========================================
# 1. LOAD DATA VÀ CẮT TỈA
# ==========================================
print("\n[1/4] Đang nạp dữ liệu...")
train_df = pd.read_parquet(PATH_TRAIN)
# Lấy dữ liệu từ giữa tháng 6 để cân bằng giữa Độ chính xác và RAM
train_df = train_df[train_df['date'] >= '2017-06-15'].copy() 

test_df = pd.read_parquet(PATH_TEST)
test_ids = test_df['id'].copy()

# ==========================================
# 2. BẢO TỒN CATEGORY & XỬ LÝ LỖI
# ==========================================
print("\n[2/4] Đang mã hóa Danh tính và Xử lý Lỗi Toán học...")
cat_cols = ['store_nbr', 'city', 'state', 'type', 'cluster', 'family']
for col in cat_cols:
    train_df[col] = train_df[col].astype('category').cat.codes
    test_df[col] = test_df[col].astype('category').cat.codes

drop_cols = ['date', 'item_nbr', 'class'] 

y_train = train_df['unit_sales']
X_train = train_df.drop(columns=drop_cols + ['unit_sales'])
# Xử lý Vô cực bằng NaN, sau đó fill toàn bộ bằng -1 để RF tự phân nhánh riêng
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(-1)

del train_df; gc.collect()

X_test = test_df.drop(columns=drop_cols + ['id'])
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(-1)
del test_df; gc.collect()

# ==========================================
# 3. TRỒNG RỪNG NGUYÊN SINH
# ==========================================
print("\n[3/4] Rừng đang mọc (Giữ nguyên sự đa dạng)...")
rf_model = RandomForestRegressor(
    n_estimators=50,        
    max_depth=14,           # Cho phép cây sâu hơn chút để khai thác 61 features
    max_features='sqrt',    # BẮT BUỘC: Giữ 'sqrt' để đảm bảo sự ngẫu nhiên của các cây
    max_samples=0.25,       # Học trên 25% dữ liệu (vừa đủ khôn, không lo OOM)
    n_jobs=-1, 
    random_state=42
)

rf_model.fit(X_train, y_train)

del X_train, y_train; gc.collect()

# ==========================================
# 4. DỰ ĐOÁN & XUẤT FILE
# ==========================================
print("\n[4/4] Đang dự đoán và xuất xưởng...")
preds_log = rf_model.predict(X_test)

# DỊCH NGƯỢC: Cực kỳ quan trọng (Giả định unit_sales trong file đã log1p)
preds_real = np.expm1(preds_log) 
preds_real = np.clip(preds_real, 0, None) 

submission = pd.DataFrame({'id': test_ids, 'unit_sales': preds_real})
submission.to_csv('submission_rf_redemption.csv', index=False, float_format='%.4f')

print(f"\n=====================================================")
print(f"✅ HOÀN TẤT BẢN CHUỘC LỖI! ({(time.time() - start_time)/60:.2f} phút)")
print(f"Đã trả lại sự nguyên bản cho Random Forest!")
print(f"=====================================================")

🚀 CHIẾN DỊCH CHUỘC LỖI: FULL FEATURES + CHỐNG OOM TỐI ƯU 🚀

[1/4] Đang nạp dữ liệu...

[2/4] Đang mã hóa Danh tính và Xử lý Lỗi Toán học...

[3/4] Rừng đang mọc (Giữ nguyên sự đa dạng)...

[4/4] Đang dự đoán và xuất xưởng...

✅ HOÀN TẤT BẢN CHUỘC LỖI! (5.03 phút)
Đã trả lại sự nguyên bản cho Random Forest!
